# Notebook 03 — Structural Harmonization

This notebook loads MedQuAD, MedDialog-EN, and HealthChat-11K, inspects their actual schemas,
and converts all three into a common turn-level structure.

**Important findings from prior inspection:**
- **MedQuAD** (`medquad.csv`): 16,412 rows, columns: `question`, `answer`, `source`, `focus_area`. Single-turn QA pairs.
- **MedDialog-EN** (`meddialog/`): 112,165 rows, columns: `instruction`, `input`, `output`. Single-turn instruction-format pairs from a doctor-patient QA collection.
- **HealthChat-11K** (`health11k/`): 33,022 rows, columns: `conversation_id`, `dataset_source`, `specialty_conversation_classification`, `taxonomy_messages_classified`, `web_url`, `gpt2_tokenizer_count`, `leading_question_classifications`. **This dataset contains metadata/annotation only — no raw utterance text.** The conversations originate from lmsys-chat-1m and WildChat and would need to be fetched separately. The taxonomy codes and specialty labels are preserved as `source_label_raw`.


In [1]:
import pandas as pd
import numpy as np
import json
import os
import warnings
from pathlib import Path
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
tqdm.pandas()

# Paths
RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed/structural')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('Output directory:', PROCESSED_DIR.resolve())

Output directory: C:\Users\nirmi\Desktop\Capstone\data\processed\structural


## 1. MedQuAD — Schema Inspection

In [2]:
medquad_path = RAW_DIR / 'medquad.csv'
medquad_raw = pd.read_csv(medquad_path)

print('=== MedQuAD ===' )
print('Shape:', medquad_raw.shape)
print('Columns:', medquad_raw.columns.tolist())
print()
print('Dtypes:')
print(medquad_raw.dtypes)
print()
print('Missing values:')
print(medquad_raw.isnull().sum())
print()
print('First 5 records:')
display(medquad_raw.head())

=== MedQuAD ===
Shape: (16412, 4)
Columns: ['question', 'answer', 'source', 'focus_area']

Dtypes:
question      str
answer        str
source        str
focus_area    str
dtype: object

Missing values:
question       0
answer         5
source         0
focus_area    14
dtype: int64

First 5 records:


,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma


In [3]:
print('=== MedQuAD: Available labels (focus_area) ===')
print(f'Unique focus_area values: {medquad_raw["focus_area"].nunique()}')
print(medquad_raw['focus_area'].value_counts().head(20))
print()
print('=== MedQuAD: Available labels (source) ===')
print(medquad_raw['source'].value_counts())
print()
print('=== Example record ===')
ex = medquad_raw.iloc[0]
print('Question:', ex['question'])
print('Answer:', ex['answer'][:300] if pd.notna(ex['answer']) else 'NaN')
print('Source:', ex['source'])
print('Focus area:', ex['focus_area'])

=== MedQuAD: Available labels (focus_area) ===
Unique focus_area values: 5126
focus_area
Breast Cancer                       53
Prostate Cancer                     43
Stroke                              35
Skin Cancer                         34
Alzheimer's Disease                 30
Colorectal Cancer                   29
Lung Cancer                         29
High Blood Cholesterol              28
Heart Attack                        28
Heart Failure                       28
Causes of Diabetes                  28
High Blood Pressure                 27
Parkinson's Disease                 25
Leukemia                            22
Osteoporosis                        21
Shingles                            21
Diabetes                            20
Age-related Macular Degeneration    20
Hemochromatosis                     20
Diabetic Retinopathy                19
Name: count, dtype: int64

=== MedQuAD: Available labels (source) ===
source
GHR                  5430
GARD                 5394
NI

## 2. MedDialog-EN — Schema Inspection

In [4]:
from datasets import load_from_disk

meddialog_path = RAW_DIR / 'meddialog'
meddialog_ds = load_from_disk(str(meddialog_path))

print('=== MedDialog-EN ===')
print('Dataset:', meddialog_ds)
print('Features:', meddialog_ds['train'].features)
print()

meddialog_df = meddialog_ds['train'].to_pandas()
print('Shape:', meddialog_df.shape)
print('Columns:', meddialog_df.columns.tolist())
print()
print('Missing values:')
print(meddialog_df.isnull().sum())
print()
print('First 5 records:')
display(meddialog_df.head())

=== MedDialog-EN ===
Dataset: DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 112165
    })
})
Features: {'instruction': Value('string'), 'input': Value('string'), 'output': Value('string')}

Shape: (112165, 3)
Columns: ['instruction', 'input', 'output']

Missing values:
instruction    0
input          0
output         0
dtype: int64

First 5 records:


,instruction,input,output
0,"If you are a doctor, please answer the medical...",I woke up this morning feeling the whole room ...,"Hi, Thank you for posting your query. The most..."
1,"If you are a doctor, please answer the medical...",My baby has been pooing 5-6 times a day for a ...,Hi... Thank you for consulting in Chat Doctor....
2,"If you are a doctor, please answer the medical...","Hello, My husband is taking Oxycodone due to a...","Hello, and I hope I can help you today.First, ..."
3,"If you are a doctor, please answer the medical...",lump under left nipple and stomach pain (male)...,HI. You have two different problems. The lump ...
4,"If you are a doctor, please answer the medical...",I have a 5 month old baby who is very congeste...,Thank you for using Chat Doctor. I would sugge...


In [5]:
print('=== MedDialog: Available labels ===')
print('Unique instruction values:', meddialog_df['instruction'].nunique())
print(meddialog_df['instruction'].value_counts())
print()
print('=== Example dialogue ===')
ex = meddialog_df.iloc[0]
print('Instruction:', ex['instruction'])
print('Input:', ex['input'][:400])
print('Output:', ex['output'][:400])

=== MedDialog: Available labels ===
Unique instruction values: 1
instruction
If you are a doctor, please answer the medical questions based on the patient's description.    112165
Name: count, dtype: int64

=== Example dialogue ===
Instruction: If you are a doctor, please answer the medical questions based on the patient's description.
Input: I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinni
Output: Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo (BPPV), a type of peripheral vertigo. In this condition, the most common symptom is dizziness or giddiness, which is made worse 

## 3. HealthChat-11K — Schema Inspection

In [6]:
import pyarrow.ipc as ipc

health11k_path = RAW_DIR / 'health11k'
health11k_ds = load_from_disk(str(health11k_path))

print('=== HealthChat-11K ===')
print('Dataset:', health11k_ds)
print('Features:', health11k_ds['train'].features)
print()

health11k_df = health11k_ds['train'].to_pandas()
print('Shape:', health11k_df.shape)
print('Columns:', health11k_df.columns.tolist())
print()
print('Missing values:')
print(health11k_df.isnull().sum())
print()
print('First 5 records:')
display(health11k_df.head())

=== HealthChat-11K ===
Dataset: DatasetDict({
    train: Dataset({
        features: ['conversation_id', 'dataset_source', 'specialty_conversation_classification', 'taxonomy_messages_classified', 'web_url', 'gpt2_tokenizer_count', 'leading_question_classifications'],
        num_rows: 33022
    })
})
Features: {'conversation_id': Value('string'), 'dataset_source': Value('string'), 'specialty_conversation_classification': Value('int64'), 'taxonomy_messages_classified': List({'taxonomy_codes': List(Value('string'))}), 'web_url': Value('string'), 'gpt2_tokenizer_count': Value('int64'), 'leading_question_classifications': List({'user_message_original_turn_index': Value('int64'), 'classification': Value('string')})}

Shape: (33022, 7)
Columns: ['conversation_id', 'dataset_source', 'specialty_conversation_classification', 'taxonomy_messages_classified', 'web_url', 'gpt2_tokenizer_count', 'leading_question_classifications']

Missing values:
conversation_id                              0
datas

,conversation_id,dataset_source,specialty_conversation_classification,taxonomy_messages_classified,web_url,gpt2_tokenizer_count,leading_question_classifications
0,000573958699464e9de6493b5e182fab,lmsys,2,"[{'taxonomy_codes': ['A1.1', 'B5.2']}, {'taxon...",https://wildvisualizer.com/conversation/lmsysc...,994,"[{'user_message_original_turn_index': 0, 'clas..."
1,000ac740005818a54be24a9d8ffb8e49,wildchat,16,"[{'taxonomy_codes': ['A1.1', 'B5.2']}]",https://wildvisualizer.com/conversation/wildch...,636,"[{'user_message_original_turn_index': 0, 'clas..."
2,00110769d51fdd16b174e05e6407c50a,wildchat,4,[{'taxonomy_codes': ['B5.1.2']}],https://wildvisualizer.com/conversation/wildch...,301,"[{'user_message_original_turn_index': 0, 'clas..."
3,0013a16e8e9064969120c69ec83c51b5,wildchat,21,[{'taxonomy_codes': ['B9']}],https://wildvisualizer.com/conversation/wildch...,313,"[{'user_message_original_turn_index': 0, 'clas..."
4,00187dceda744814a3f59510c9256827,lmsys,2,[{'taxonomy_codes': ['B3.3']}],https://wildvisualizer.com/conversation/lmsysc...,339,None


In [7]:
print('=== HealthChat-11K: dataset_source breakdown ===')
print(health11k_df['dataset_source'].value_counts())
print()
print('=== HealthChat-11K: specialty_conversation_classification ===')
print(health11k_df['specialty_conversation_classification'].value_counts().sort_index())
print()
print('=== HealthChat-11K: IMPORTANT NOTE ===')
print("""
The local HealthChat-11K download contains METADATA ONLY — no raw utterance text.
The actual conversation text comes from:
  - lmsys-chat-1m (20,388 conversations)
  - WildChat (12,634 conversations)
These conversations can be fetched from their respective HuggingFace datasets or via the web_url.

What IS available locally:
  - conversation_id: unique identifier (links to lmsys/wildchat)
  - dataset_source: 'lmsys' or 'wildchat'
  - specialty_conversation_classification: numeric specialty label (1-22)
  - taxonomy_messages_classified: per-turn taxonomy codes (e.g. A1.1, B5.2)
  - web_url: URL to view the conversation at wildvisualizer.com
  - gpt2_tokenizer_count: token count of the conversation
  - leading_question_classifications: which user turns contain leading questions

Since utterance text is unavailable without fetching external datasets,
HealthChat-11K will be represented as METADATA RECORDS in the structural output.
utterance will be set to None/NaN with a flag indicating the text must be resolved.
""")

print('=== Example record ===')
ex = health11k_df.iloc[0]
print('conversation_id:', ex['conversation_id'])
print('dataset_source:', ex['dataset_source'])
print('specialty:', ex['specialty_conversation_classification'])
print('taxonomy_messages_classified:', ex['taxonomy_messages_classified'])
print('web_url:', ex['web_url'])
print('gpt2_tokenizer_count:', ex['gpt2_tokenizer_count'])
print('leading_question_classifications:', ex['leading_question_classifications'])

=== HealthChat-11K: dataset_source breakdown ===
dataset_source
lmsys       20388
wildchat    12634
Name: count, dtype: int64

=== HealthChat-11K: specialty_conversation_classification ===
specialty_conversation_classification
1     8995
2     4196
3      341
4     1039
5     1042
6      319
7     1024
8     1462
9      713
10     313
11    1211
12     783
13     463
14    2823
15     465
16     662
17     574
18     231
19     498
20     556
21    4670
22     642
Name: count, dtype: int64

=== HealthChat-11K: IMPORTANT NOTE ===

The local HealthChat-11K download contains METADATA ONLY — no raw utterance text.
The actual conversation text comes from:
  - lmsys-chat-1m (20,388 conversations)
  - WildChat (12,634 conversations)
These conversations can be fetched from their respective HuggingFace datasets or via the web_url.

What IS available locally:
  - conversation_id: unique identifier (links to lmsys/wildchat)
  - dataset_source: 'lmsys' or 'wildchat'
  - specialty_conversation_clas

In [8]:
# Summarize taxonomy codes across the dataset
all_codes = []
for row in health11k_df['taxonomy_messages_classified']:
    if row is not None:
        for msg in row:
            if msg is not None and 'taxonomy_codes' in msg:
                all_codes.extend(msg['taxonomy_codes'])

print('=== Top 25 taxonomy codes across all turns ===')
print(pd.Series(all_codes).value_counts().head(25))
print()
print(f'Total taxonomy code assignments: {len(all_codes)}')
print(f'Unique taxonomy codes: {pd.Series(all_codes).nunique()}')

=== Top 25 taxonomy codes across all turns ===
B9      12950
B5.2    10599
A1.1     7491
B3.3     6903
B8       6163
B10      5806
D1       5092
B1       4917
B5.1     4492
B8.2     4290
B4       3667
B2       2867
C6       2835
C1       2636
B7       2290
A1.2     2101
A1.5     1990
A1.7     1880
A1.3     1878
B8.1     1802
B3.1     1306
A2.4     1239
C2       1076
B3.4     1074
B3.2     1041
Name: count, dtype: int64

Total taxonomy code assignments: 105928


Unique taxonomy codes: 48


## 4. Structural Harmonization

Convert all three datasets into the common turn-level schema:

| Column | Description |
|---|---|
| `dialogue_id` | Globally unique conversation identifier |
| `turn_id` | Turn number within the conversation |
| `speaker` | `user` or `assistant` |
| `utterance` | Actual text (None if unavailable) |
| `source_dataset` | `MedQuAD`, `MedDialog`, or `HealthChat` |
| `original_id` | Original source record/conversation identifier |
| `source_label_raw` | Original label/category when available |
| `dialogue_origin` | `original` or `constructed` |

### 4a. MedQuAD → Turn-level (constructed dialogues from QA pairs)

In [9]:
def harmonize_medquad(df):
    """
    Each MedQuAD row is a QA pair.
    Convert to 2-turn constructed dialogue:
      turn 0: speaker=user,      utterance=question
      turn 1: speaker=assistant, utterance=answer
    Rows with null answer are kept but utterance will be None.
    """
    records = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc='MedQuAD'):
        dialogue_id = f'medquad_{idx:06d}'
        original_id = str(idx)
        source_label = row['focus_area'] if pd.notna(row.get('focus_area')) else None

        # Turn 0 — user question
        records.append({
            'dialogue_id': dialogue_id,
            'turn_id': 0,
            'speaker': 'user',
            'utterance': str(row['question']) if pd.notna(row['question']) else None,
            'source_dataset': 'MedQuAD',
            'original_id': original_id,
            'source_label_raw': source_label,
            'dialogue_origin': 'constructed',
        })

        # Turn 1 — assistant answer
        records.append({
            'dialogue_id': dialogue_id,
            'turn_id': 1,
            'speaker': 'assistant',
            'utterance': str(row['answer']) if pd.notna(row['answer']) else None,
            'source_dataset': 'MedQuAD',
            'original_id': original_id,
            'source_label_raw': source_label,
            'dialogue_origin': 'constructed',
        })

    return pd.DataFrame(records)

medquad_turns = harmonize_medquad(medquad_raw)
print('MedQuAD turns shape:', medquad_turns.shape)
print('Sample:')
display(medquad_turns.head(6))

MedQuAD:   0%|          | 0/16412 [00:00<?, ?it/s]

MedQuAD turns shape: (32824, 8)
Sample:


,dialogue_id,turn_id,speaker,utterance,source_dataset,original_id,source_label_raw,dialogue_origin
0,medquad_000000,0,user,What is (are) Glaucoma ?,MedQuAD,0,Glaucoma,constructed
1,medquad_000000,1,assistant,Glaucoma is a group of diseases that can damag...,MedQuAD,0,Glaucoma,constructed
2,medquad_000001,0,user,What causes Glaucoma ?,MedQuAD,1,Glaucoma,constructed
3,medquad_000001,1,assistant,"Nearly 2.7 million people have glaucoma, a lea...",MedQuAD,1,Glaucoma,constructed
4,medquad_000002,0,user,What are the symptoms of Glaucoma ?,MedQuAD,2,Glaucoma,constructed
5,medquad_000002,1,assistant,Symptoms of Glaucoma Glaucoma can develop in ...,MedQuAD,2,Glaucoma,constructed


### 4b. MedDialog-EN → Turn-level (constructed dialogues from instruction/input/output)

In [10]:
def harmonize_meddialog(df):
    """
    Each MedDialog row is an instruction/input/output triple.
    The instruction is a system prompt, not a user utterance.
    Convert to 2-turn constructed dialogue:
      turn 0: speaker=user,      utterance=input
      turn 1: speaker=assistant, utterance=output
    The instruction is preserved in source_label_raw.
    """
    records = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc='MedDialog'):
        dialogue_id = f'meddialog_{idx:07d}'
        original_id = str(idx)
        # Instruction is the same for all rows — preserve it as label
        source_label = str(row['instruction'])[:200] if pd.notna(row.get('instruction')) else None

        # Turn 0 — user (patient query)
        records.append({
            'dialogue_id': dialogue_id,
            'turn_id': 0,
            'speaker': 'user',
            'utterance': str(row['input']) if pd.notna(row['input']) else None,
            'source_dataset': 'MedDialog',
            'original_id': original_id,
            'source_label_raw': source_label,
            'dialogue_origin': 'constructed',
        })

        # Turn 1 — assistant (doctor response)
        records.append({
            'dialogue_id': dialogue_id,
            'turn_id': 1,
            'speaker': 'assistant',
            'utterance': str(row['output']) if pd.notna(row['output']) else None,
            'source_dataset': 'MedDialog',
            'original_id': original_id,
            'source_label_raw': source_label,
            'dialogue_origin': 'constructed',
        })

    return pd.DataFrame(records)

meddialog_turns = harmonize_meddialog(meddialog_df)
print('MedDialog turns shape:', meddialog_turns.shape)
print('Sample:')
display(meddialog_turns.head(6))

MedDialog:   0%|          | 0/112165 [00:00<?, ?it/s]

MedDialog turns shape: (224330, 8)
Sample:


,dialogue_id,turn_id,speaker,utterance,source_dataset,original_id,source_label_raw,dialogue_origin
0,meddialog_0000000,0,user,I woke up this morning feeling the whole room ...,MedDialog,0,"If you are a doctor, please answer the medical...",constructed
1,meddialog_0000000,1,assistant,"Hi, Thank you for posting your query. The most...",MedDialog,0,"If you are a doctor, please answer the medical...",constructed
2,meddialog_0000001,0,user,My baby has been pooing 5-6 times a day for a ...,MedDialog,1,"If you are a doctor, please answer the medical...",constructed
3,meddialog_0000001,1,assistant,Hi... Thank you for consulting in Chat Doctor....,MedDialog,1,"If you are a doctor, please answer the medical...",constructed
4,meddialog_0000002,0,user,"Hello, My husband is taking Oxycodone due to a...",MedDialog,2,"If you are a doctor, please answer the medical...",constructed
5,meddialog_0000002,1,assistant,"Hello, and I hope I can help you today.First, ...",MedDialog,2,"If you are a doctor, please answer the medical...",constructed


### 4c. HealthChat-11K → Turn-level (metadata records; utterance text unavailable)

Since the local HealthChat-11K download does not contain conversation text, each row is represented
as a metadata-level dialogue record. Per-turn taxonomy codes are expanded into individual turns
with alternating speaker assignment (user/assistant), matching the conversational structure
implied by the turn index in `leading_question_classifications`.

The `utterance` field is `None` for all HealthChat records. The taxonomy codes per turn
are stored as `source_label_raw`.

In [11]:
# Specialty code → human-readable label mapping
# Based on HealthChat-11K paper / taxonomy documentation
SPECIALTY_MAP = {
    1: 'General Medicine',
    2: 'Mental Health',
    3: 'Dermatology',
    4: 'Cardiology',
    5: 'Neurology',
    6: 'Oncology',
    7: 'Gastroenterology',
    8: 'Orthopedics',
    9: 'Endocrinology',
    10: 'Pulmonology',
    11: 'Pediatrics',
    12: 'Gynecology',
    13: 'Urology',
    14: 'Ophthalmology',
    15: 'Otolaryngology',
    16: 'Rheumatology',
    17: 'Hematology',
    18: 'Nephrology',
    19: 'Infectious Disease',
    20: 'Emergency Medicine',
    21: 'Other',
    22: 'Sexual Health',
}

def harmonize_health11k(df):
    """
    HealthChat-11K has no utterance text locally.
    Expand per-turn taxonomy codes into turn-level records.
    Speakers alternate: turn 0 = user, turn 1 = assistant, etc.
    (Standard for LMSYS/WildChat which are alternating human/gpt conversations.)
    """
    records = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc='HealthChat-11K'):
        dialogue_id = f'healthchat_{idx:06d}'
        original_id = str(row['conversation_id'])
        specialty_code = int(row['specialty_conversation_classification'])
        specialty_label = SPECIALTY_MAP.get(specialty_code, f'specialty_{specialty_code}')
        dataset_source_origin = str(row['dataset_source'])  # 'lmsys' or 'wildchat'

        taxonomy_turns = row['taxonomy_messages_classified']
        leading_q = row['leading_question_classifications']

        if taxonomy_turns is None or len(taxonomy_turns) == 0:
            # No turn info available — create a single placeholder dialogue record
            records.append({
                'dialogue_id': dialogue_id,
                'turn_id': 0,
                'speaker': 'user',
                'utterance': None,
                'source_dataset': 'HealthChat',
                'original_id': original_id,
                'source_label_raw': json.dumps({
                    'specialty': specialty_label,
                    'specialty_code': specialty_code,
                    'dataset_source': dataset_source_origin,
                    'taxonomy_codes': [],
                }),
                'dialogue_origin': 'original',
            })
            continue

        # Build set of leading question turn indices for this dialogue
        leading_q_indices = set()
        if leading_q is not None:
            for item in leading_q:
                if item is not None:
                    leading_q_indices.add(item['user_message_original_turn_index'])

        for turn_idx, turn_info in enumerate(taxonomy_turns):
            # LMSYS/WildChat: even turns = human (user), odd turns = assistant
            speaker = 'user' if turn_idx % 2 == 0 else 'assistant'

            codes = []
            if turn_info is not None and 'taxonomy_codes' in turn_info:
                codes = list(turn_info['taxonomy_codes'])

            records.append({
                'dialogue_id': dialogue_id,
                'turn_id': turn_idx,
                'speaker': speaker,
                'utterance': None,  # Text unavailable locally
                'source_dataset': 'HealthChat',
                'original_id': original_id,
                'source_label_raw': json.dumps({
                    'specialty': specialty_label,
                    'specialty_code': specialty_code,
                    'dataset_source': dataset_source_origin,
                    'taxonomy_codes': codes,
                    'is_leading_question': (turn_idx in leading_q_indices),
                }),
                'dialogue_origin': 'original',
            })

    return pd.DataFrame(records)

health11k_turns = harmonize_health11k(health11k_df)
print('HealthChat-11K turns shape:', health11k_turns.shape)
print('Sample:')
display(health11k_turns.head(10))

HealthChat-11K:   0%|          | 0/33022 [00:00<?, ?it/s]

HealthChat-11K turns shape: (76879, 8)
Sample:


,dialogue_id,turn_id,speaker,utterance,source_dataset,original_id,source_label_raw,dialogue_origin
0,healthchat_000000,0,user,None,HealthChat,000573958699464e9de6493b5e182fab,"{""specialty"": ""Mental Health"", ""specialty_code...",original
1,healthchat_000000,1,assistant,None,HealthChat,000573958699464e9de6493b5e182fab,"{""specialty"": ""Mental Health"", ""specialty_code...",original
2,healthchat_000000,2,user,None,HealthChat,000573958699464e9de6493b5e182fab,"{""specialty"": ""Mental Health"", ""specialty_code...",original
3,healthchat_000000,3,assistant,None,HealthChat,000573958699464e9de6493b5e182fab,"{""specialty"": ""Mental Health"", ""specialty_code...",original
4,healthchat_000000,4,user,None,HealthChat,000573958699464e9de6493b5e182fab,"{""specialty"": ""Mental Health"", ""specialty_code...",original
5,healthchat_000001,0,user,None,HealthChat,000ac740005818a54be24a9d8ffb8e49,"{""specialty"": ""Rheumatology"", ""specialty_code""...",original
6,healthchat_000002,0,user,None,HealthChat,00110769d51fdd16b174e05e6407c50a,"{""specialty"": ""Cardiology"", ""specialty_code"": ...",original
7,healthchat_000003,0,user,None,HealthChat,0013a16e8e9064969120c69ec83c51b5,"{""specialty"": ""Other"", ""specialty_code"": 21, ""...",original
8,healthchat_000004,0,user,None,HealthChat,00187dceda744814a3f59510c9256827,"{""specialty"": ""Mental Health"", ""specialty_code...",original
9,healthchat_000005,0,user,None,HealthChat,001e1642a7c80342c0ca3653c52e16a4,"{""specialty"": ""Other"", ""specialty_code"": 21, ""...",original


## 5. Concatenate and Save

In [12]:
# Concatenate all three
harmonized = pd.concat(
    [medquad_turns, meddialog_turns, health11k_turns],
    ignore_index=True
)

# Enforce column order
STRUCTURAL_COLS = [
    'dialogue_id', 'turn_id', 'speaker', 'utterance',
    'source_dataset', 'original_id', 'source_label_raw', 'dialogue_origin'
]
harmonized = harmonized[STRUCTURAL_COLS]

print('=== Combined structural dataset ===')
print('Shape:', harmonized.shape)
print('Dtypes:')
print(harmonized.dtypes)
print()
print('Sample rows:')
display(harmonized.head(10))

=== Combined structural dataset ===
Shape: (334033, 8)
Dtypes:
dialogue_id            str
turn_id              int64
speaker                str
utterance           object
source_dataset         str
original_id            str
source_label_raw       str
dialogue_origin        str
dtype: object

Sample rows:


,dialogue_id,turn_id,speaker,utterance,source_dataset,original_id,source_label_raw,dialogue_origin
0,medquad_000000,0,user,What is (are) Glaucoma ?,MedQuAD,0,Glaucoma,constructed
1,medquad_000000,1,assistant,Glaucoma is a group of diseases that can damag...,MedQuAD,0,Glaucoma,constructed
2,medquad_000001,0,user,What causes Glaucoma ?,MedQuAD,1,Glaucoma,constructed
3,medquad_000001,1,assistant,"Nearly 2.7 million people have glaucoma, a lea...",MedQuAD,1,Glaucoma,constructed
4,medquad_000002,0,user,What are the symptoms of Glaucoma ?,MedQuAD,2,Glaucoma,constructed
5,medquad_000002,1,assistant,Symptoms of Glaucoma Glaucoma can develop in ...,MedQuAD,2,Glaucoma,constructed
6,medquad_000003,0,user,What are the treatments for Glaucoma ?,MedQuAD,3,Glaucoma,constructed
7,medquad_000003,1,assistant,"Although open-angle glaucoma cannot be cured, ...",MedQuAD,3,Glaucoma,constructed
8,medquad_000004,0,user,What is (are) Glaucoma ?,MedQuAD,4,Glaucoma,constructed
9,medquad_000004,1,assistant,Glaucoma is a group of diseases that can damag...,MedQuAD,4,Glaucoma,constructed


In [13]:
# Ensure dialogue_id + turn_id uniqueness
dups = harmonized.duplicated(subset=['dialogue_id', 'turn_id']).sum()
print(f'Duplicate (dialogue_id, turn_id) pairs: {dups}')
assert dups == 0, 'Duplicate keys found — check harmonization logic'

# Check speaker values
print('Speaker values:', harmonized['speaker'].value_counts().to_dict())
assert set(harmonized['speaker'].unique()) <= {'user', 'assistant'}, 'Unexpected speaker values'

# Check dialogue_origin values
print('dialogue_origin values:', harmonized['dialogue_origin'].value_counts().to_dict())
assert set(harmonized['dialogue_origin'].unique()) <= {'original', 'constructed'}

print('All structural checks passed.')

Duplicate (dialogue_id, turn_id) pairs: 0
Speaker values: {'user': 179046, 'assistant': 154987}
dialogue_origin values: {'constructed': 257154, 'original': 76879}
All structural checks passed.


In [14]:
# Save
parquet_path = PROCESSED_DIR / 'harmonized_structural.parquet'
csv_path = PROCESSED_DIR / 'harmonized_structural.csv'

harmonized.to_parquet(parquet_path, index=False)
harmonized.to_csv(csv_path, index=False)

print(f'Saved parquet: {parquet_path}')
print(f'Saved CSV:     {csv_path}')
print(f'Parquet size:  {parquet_path.stat().st_size / 1024 / 1024:.1f} MB')
print(f'CSV size:      {csv_path.stat().st_size / 1024 / 1024:.1f} MB')

Saved parquet: ..\data\processed\structural\harmonized_structural.parquet
Saved CSV:     ..\data\processed\structural\harmonized_structural.csv
Parquet size:  82.7 MB
CSV size:      190.6 MB


## 6. Summary Statistics

In [15]:
print('=' * 60)
print('STRUCTURAL HARMONIZATION SUMMARY')
print('=' * 60)

for dataset in ['MedQuAD', 'MedDialog', 'HealthChat']:
    subset = harmonized[harmonized['source_dataset'] == dataset]
    n_dialogues = subset['dialogue_id'].nunique()
    n_turns = len(subset)
    print(f'\n{dataset}:')
    print(f'  Dialogues : {n_dialogues:,}')
    print(f'  Turns     : {n_turns:,}')

print()
total_dialogues = harmonized['dialogue_id'].nunique()
total_turns = len(harmonized)
original = harmonized[harmonized['dialogue_origin'] == 'original']['dialogue_id'].nunique()
constructed = harmonized[harmonized['dialogue_origin'] == 'constructed']['dialogue_id'].nunique()
user_turns = (harmonized['speaker'] == 'user').sum()
assistant_turns = (harmonized['speaker'] == 'assistant').sum()

print(f'Total dialogues              : {total_dialogues:,}')
print(f'Total turns                  : {total_turns:,}')
print(f'Original dialogues           : {original:,}')
print(f'Constructed dialogues        : {constructed:,}')
print(f'User turns                   : {user_turns:,}')
print(f'Assistant turns              : {assistant_turns:,}')
print()
print('Turns per dataset:')
print(harmonized['source_dataset'].value_counts())
print()
print('Dialogues per dataset:')
print(harmonized.groupby('source_dataset')['dialogue_id'].nunique())

STRUCTURAL HARMONIZATION SUMMARY

MedQuAD:
  Dialogues : 16,412
  Turns     : 32,824



MedDialog:
  Dialogues : 112,165
  Turns     : 224,330

HealthChat:
  Dialogues : 33,022
  Turns     : 76,879



Total dialogues              : 161,599
Total turns                  : 334,033
Original dialogues           : 33,022
Constructed dialogues        : 128,577
User turns                   : 179,046
Assistant turns              : 154,987

Turns per dataset:
source_dataset
MedDialog     224330
HealthChat     76879
MedQuAD        32824
Name: count, dtype: int64

Dialogues per dataset:
source_dataset
HealthChat     33022
MedDialog     112165
MedQuAD        16412
Name: dialogue_id, dtype: int64
